# 시계열 데이터 신경망 예측 튜토리얼

본 튜토리얼에서는 다음 단계를 다룹니다.

1. 필요한 라이브러리 임포트
2. 실제 시계열 데이터 (Daily Minimum Temperatures, Melbourne) 불러오기
3. 시계열 데이터를 신경망 학습용 입/출력 페어 (N x L x D 형태) 로 변환
4. PyTorch `Dataset` / `DataLoader` 구성
5. 간단한 신경망 모델 (MLP, LSTM) 정의
6. 학습 루프 구성 및 학습
7. 예측 결과 시각화

사용 데이터: **Daily Minimum Temperatures in Melbourne (1981–1990)**
- 출처: https://github.com/jbrownlee/Datasets
- 일(day) 단위, `Date` 컬럼 + 일 최저기온 1개 변수 (단변량, D=1)
- 총 3,650 일치 (약 10년)

Google Colab에서 그대로 실행할 수 있도록 작성되어 있습니다.

## 1. 라이브러리 임포트

PyTorch와 시각화·데이터 처리에 필요한 라이브러리를 불러옵니다. Colab에는 기본적으로 설치되어 있으므로 별도 설치는 필요 없습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 재현성을 위한 시드 고정
np.random.seed(0)
torch.manual_seed(0)

# GPU 사용 가능 여부 확인 (Colab에서 런타임 유형을 GPU로 바꾸면 cuda 사용 가능)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 디바이스:', device)

### 한글 폰트 설정 (Colab 환경)

코랩에는 기본적으로 한글 폰트가 없어서 matplotlib 그래프에 한글이 깨집니다. 나눔고딕을 설치하고 matplotlib에 등록합니다.

> 처음 한 번만 실행하면 됩니다. 설치 후 폰트 캐시를 갱신하므로 셀 실행이 끝나면 바로 다음 셀로 진행해도 됩니다. (간혹 안 먹으면 런타임 재시작 후 다시 실행)

In [ ]:
# 코랩에 나눔고딕 설치 후 matplotlib에 등록
!apt-get install -y -qq fonts-nanum > /dev/null
!fc-cache -fv > /dev/null

import matplotlib as mpl
import matplotlib.font_manager as fm

# 폰트 캐시 재구성 (런타임 재시작 없이 인식시키기)
for fp in fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum']):
    fm.fontManager.addfont(fp)

mpl.rc('font', family='NanumGothic')
mpl.rc('axes', unicode_minus=False)  # 마이너스 부호 깨짐 방지

# 적용 확인
print('현재 폰트:', mpl.rcParams['font.family'])

## 2. 시계열 데이터 불러오기

GitHub (`jbrownlee/Datasets`) 에 공개된 CSV 파일을 `pandas` 로 직접 읽어옵니다.

- 컬럼: `Date` (날짜), `Temp` (일 최저기온, ℃)
- 기간: 1981-01-01 ~ 1990-12-31
- 길이 T = 3650, 변수 개수 D = 1

In [ ]:
URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'

# Date 컬럼을 datetime 으로 파싱하고 인덱스로 지정
df = pd.read_csv(URL, parse_dates=['Date'], index_col='Date')

# 일부 행에 따옴표나 ? 같은 비정상 값이 섞여 있을 수 있어서 숫자로 강제 변환 후 결측 보간
df['Temp'] = pd.to_numeric(df['Temp'], errors='coerce')
df['Temp'] = df['Temp'].interpolate()

print('전체 shape:', df.shape)
print('시작/끝 날짜:', df.index.min(), '~', df.index.max())
df.head()

In [ ]:
# 전체 시계열 시각화
plt.figure(figsize=(12, 3))
plt.plot(df.index, df['Temp'])
plt.title('Daily Minimum Temperatures in Melbourne (1981-1990)')
plt.xlabel('날짜'); plt.ylabel('최저기온 (℃)')
plt.grid(True); plt.show()

### (T, D) 형태 numpy 배열로 변환

신경망에 넣기 전에 시간 인덱스는 분리하고 값만 `(T, D)` 모양의 배열로 만듭니다.
단변량이므로 D = 1 이지만, 모델 코드와의 일관성을 위해 마지막 축을 유지합니다.

In [ ]:
dates  = df.index.to_numpy()                       # (T,) — 시각화용
series = df[['Temp']].to_numpy().astype(np.float32)  # (T, D)

T, D = series.shape
print('원본 시계열 shape:', series.shape, '  T =', T, ', D =', D)

### 정규화 (Standardization)

신경망 학습 안정성을 위해 평균 0, 표준편차 1로 스케일링합니다.
**중요**: 정규화 통계는 학습 구간에서만 계산해야 검증 구간 정보 누수가 없습니다.

In [ ]:
split_t = int(0.8 * T)  # 시간축 기준 80% 지점

mean = series[:split_t].mean(axis=0)
std  = series[:split_t].std(axis=0)

series_norm = (series - mean) / std
print('정규화 후 평균/표준편차 (학습구간):',
      series_norm[:split_t].mean(), series_norm[:split_t].std())

## 3. 입/출력 페어 구성: (T, D) → (N, L, D)

신경망은 "과거 L 스텝을 보고 다음 H 스텝을 예측" 하는 식으로 학습합니다.

- 입력 윈도우 길이 `L` (예: 30일): 과거 몇 일을 모델에 입력할지
- 예측 길이 `H` (예: 7일): 다음 며칠을 예측할지 (multi-step forecasting)

슬라이딩 윈도우로 다음과 같이 페어를 만듭니다.

- 입력 X: `(N, L, D)`
- 출력 Y: `(N, H, D)`

여기서 `N = T - L - H + 1`.

In [ ]:
def make_windows(series: np.ndarray, L: int, H: int = 1):
    """(T, D) 시계열을 슬라이딩 윈도우로 잘라 (N, L, D) 입력과 (N, H, D) 출력으로 변환."""
    T_, D_ = series.shape
    N = T_ - L - H + 1
    X = np.zeros((N, L, D_), dtype=np.float32)
    Y = np.zeros((N, H, D_), dtype=np.float32)
    for i in range(N):
        X[i] = series[i : i + L]              # 과거 L 스텝
        Y[i] = series[i + L : i + L + H]      # 그 다음 H 스텝
    return X, Y

L = 30  # 과거 30일
H = 7   # 다음 7일 예측 (multi-step)

X, Y = make_windows(series_norm, L, H)
print('X shape:', X.shape)  # (N, L, D)
print('Y shape:', Y.shape)  # (N, H, D)

### 학습/검증 데이터 분할

시계열은 시간 순서가 중요하므로 무작위 셔플 대신 **시간 순으로** 앞부분을 학습용, 뒷부분을 검증용으로 사용합니다.

주의: 위에서 정규화 통계를 `split_t` 기준으로 잡았지만, 윈도우를 만들면 인덱스가 살짝 달라집니다. 여기서는 단순화를 위해 윈도우 N 개수 기준으로 다시 80% 분할합니다.

In [ ]:
split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
Y_train, Y_val = Y[:split], Y[split:]
print('학습 샘플 수:', len(X_train), '/ 검증 샘플 수:', len(X_val))

## 4. PyTorch Dataset / DataLoader 구성

`Dataset` 은 (입력, 출력) 한 쌍을 반환하는 객체이고, `DataLoader` 는 이를 배치 단위로 묶어 줍니다.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, Y):
        # numpy array를 torch tensor로 변환
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

train_ds = TimeSeriesDataset(X_train, Y_train)
val_ds   = TimeSeriesDataset(X_val,   Y_val)

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

# 한 배치만 꺼내서 shape 확인
xb, yb = next(iter(train_loader))
print('배치 입력 shape:', xb.shape)  # (batch, L, D)
print('배치 출력 shape:', yb.shape)  # (batch, H, D)

## 5. 모델 정의

두 가지 베이스라인 모델을 정의합니다.

### 5-1. MLP 모델

`(L, D)` 입력을 1차원으로 펼쳐서 fully-connected 레이어를 거치는 단순한 모델입니다.

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, L, D, H, hidden=64):
        super().__init__()
        self.L, self.D, self.H = L, D, H
        self.net = nn.Sequential(
            nn.Linear(L * D, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, H * D),
        )

    def forward(self, x):  # x: (B, L, D)
        B = x.size(0)
        x = x.reshape(B, -1)                    # (B, L*D)
        out = self.net(x)                       # (B, H*D)
        return out.reshape(B, self.H, self.D)   # (B, H, D)

### 5-2. LSTM 모델

순환 신경망은 시계열의 시간 구조를 자연스럽게 활용합니다. 마지막 시점의 hidden state로 미래 값을 예측합니다.

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, D, H, hidden=64, num_layers=1):
        super().__init__()
        self.D, self.H = D, H
        self.lstm = nn.LSTM(input_size=D, hidden_size=hidden,
                            num_layers=num_layers, batch_first=True)
        self.head = nn.Linear(hidden, H * D)

    def forward(self, x):  # x: (B, L, D)
        out, _ = self.lstm(x)           # out: (B, L, hidden)
        last = out[:, -1, :]             # 마지막 시점만 사용: (B, hidden)
        y = self.head(last)              # (B, H*D)
        return y.reshape(x.size(0), self.H, self.D)

## 6. 학습 루프

손실 함수는 회귀 문제이므로 평균제곱오차(MSE)를 사용하고, 옵티마이저는 Adam을 사용합니다.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []
    for epoch in range(1, epochs + 1):
        # ----- 학습 -----
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
        train_loss = running / len(train_loader.dataset)

        # ----- 검증 -----
        model.eval()
        running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = criterion(pred, yb)
                running += loss.item() * xb.size(0)
        val_loss = running / len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f'[Epoch {epoch:02d}] train_loss={train_loss:.4f}  val_loss={val_loss:.4f}')

    return train_losses, val_losses

### 6-1. MLP 모델 학습

In [ ]:
mlp = MLPModel(L=L, D=D, H=H, hidden=64)
mlp_train_losses, mlp_val_losses = train_model(mlp, train_loader, val_loader, epochs=20, lr=1e-3)

### 6-2. LSTM 모델 학습

In [ ]:
lstm = LSTMModel(D=D, H=H, hidden=64, num_layers=1)
lstm_train_losses, lstm_val_losses = train_model(lstm, train_loader, val_loader, epochs=20, lr=1e-3)

### 6-3. 학습 곡선 비교

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(mlp_train_losses,  label='MLP train')
plt.plot(mlp_val_losses,    label='MLP val')
plt.plot(lstm_train_losses, label='LSTM train')
plt.plot(lstm_val_losses,   label='LSTM val')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('학습 곡선 비교')
plt.legend(); plt.grid(True); plt.show()

## 7. 예측 결과 시각화

검증 셋에 대해 모델 예측값과 실제값을 비교해 봅니다. 정규화 했었으므로 다시 **원래 스케일(℃)** 로 복원해서 그려 봅니다.

H=7 이므로 예측값은 `(N_val, 7, D)` 모양입니다. 아래 세 가지 관점으로 살펴봅니다.

1. **t+1 시계열 비교**: 각 윈도우의 \"다음 날\" 예측만 모아 시계열로 비교
2. **샘플별 7일 예측 궤적**: 몇 개 윈도우를 골라 7일 예측 곡선을 직접 비교
3. **horizon별 오차**: h=1..7 각각의 RMSE 변화 (멀어질수록 보통 어려워짐)

In [ ]:
def predict_all(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = model(xb).cpu().numpy()
            preds.append(pred)
            trues.append(yb.numpy())
    return np.concatenate(preds, axis=0), np.concatenate(trues, axis=0)

mlp_pred,  y_true = predict_all(mlp,  val_loader)
lstm_pred, _      = predict_all(lstm, val_loader)

# 정규화 복원
mlp_pred_orig  = mlp_pred  * std + mean
lstm_pred_orig = lstm_pred * std + mean
y_true_orig    = y_true    * std + mean

print('예측 shape:', mlp_pred.shape, '실제 shape:', y_true.shape)

In [ ]:
# (1) t+1 시계열 비교: 각 윈도우의 첫 예측 스텝(h=0, 즉 다음 날)만 모음
val_start_in_X = split                       # 윈도우 단위에서의 검증 시작 인덱스
val_date_start = val_start_in_X + L          # 실제 시계열에서의 날짜 인덱스 (t+1 기준)
val_dates = dates[val_date_start : val_date_start + len(y_true)]

plt.figure(figsize=(12, 4))
plt.plot(val_dates, y_true_orig[:, 0, 0],    label='실제값 (t+1)',   linewidth=1.5)
plt.plot(val_dates, mlp_pred_orig[:, 0, 0],  label='MLP 예측 (t+1)',  alpha=0.8)
plt.plot(val_dates, lstm_pred_orig[:, 0, 0], label='LSTM 예측 (t+1)', alpha=0.8)
plt.title('검증셋 t+1 예측 비교 (Melbourne 일 최저기온)')
plt.xlabel('날짜'); plt.ylabel('최저기온 (℃)')
plt.legend(); plt.grid(True); plt.show()

In [ ]:
# (2) 샘플별 [과거 30일 입력 + 미래 7일 예측] 함께 시각화
sample_indices = [0, 100, 300, 500]  # 검증셋 내 윈도우 인덱스

# 검증셋 입력(X_val)도 원 스케일로 복원해야 같이 그릴 수 있음
X_val_orig = X_val * std + mean   # (N_val, L, D)

past_x   = np.arange(-L + 1, 1)         # -29, -28, ..., 0  (과거 30일, 0이 \"오늘\")
future_x = np.arange(1, H + 1)          #  1, 2, ..., 7     (예측 horizon)

fig, axes = plt.subplots(1, len(sample_indices), figsize=(4.5 * len(sample_indices), 3.8), sharey=True)
for ax, idx in zip(axes, sample_indices):
    # 과거 30일 (입력)
    ax.plot(past_x, X_val_orig[idx, :, 0], 'k-', label='과거 입력 (L=30)', linewidth=1.5)

    # 과거 마지막 점과 미래 첫 점을 잇기 위해 시작점 추가
    last_obs = X_val_orig[idx, -1, 0]
    join_x = np.concatenate([[0], future_x])
    ax.plot(join_x, np.concatenate([[last_obs], y_true_orig[idx, :, 0]]),
            'o-',  color='tab:blue',   label='실제 미래 (H=7)')
    ax.plot(join_x, np.concatenate([[last_obs], mlp_pred_orig[idx, :, 0]]),
            's--', color='tab:orange', label='MLP 예측',  alpha=0.9)
    ax.plot(join_x, np.concatenate([[last_obs], lstm_pred_orig[idx, :, 0]]),
            '^--', color='tab:green',  label='LSTM 예측', alpha=0.9)

    ax.axvline(0, color='gray', linestyle=':', linewidth=1)  # 과거/미래 경계
    date_str = str(val_dates[idx])[:10] if idx < len(val_dates) else f'idx={idx}'
    ax.set_title(f'예측 시작일 {date_str}')
    ax.set_xlabel('상대 시점 (일,  0 = 마지막 관측일)')
    ax.grid(True)

axes[0].set_ylabel('최저기온 (℃)')
axes[0].legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# (3) horizon별 + 전체 RMSE / MAE
def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def mae(a, b):  return float(np.mean(np.abs(a - b)))

print(f'[전체 평균]  MLP   RMSE = {rmse(mlp_pred_orig,  y_true_orig):.3f} ℃,  MAE = {mae(mlp_pred_orig,  y_true_orig):.3f} ℃')
print(f'[전체 평균]  LSTM  RMSE = {rmse(lstm_pred_orig, y_true_orig):.3f} ℃,  MAE = {mae(lstm_pred_orig, y_true_orig):.3f} ℃')
print()

# horizon별 RMSE 계산
mlp_rmse_h  = [rmse(mlp_pred_orig[:, h, :],  y_true_orig[:, h, :]) for h in range(H)]
lstm_rmse_h = [rmse(lstm_pred_orig[:, h, :], y_true_orig[:, h, :]) for h in range(H)]

print('horizon  MLP_RMSE  LSTM_RMSE')
for h in range(H):
    print(f'  t+{h+1}    {mlp_rmse_h[h]:.3f}     {lstm_rmse_h[h]:.3f}')

plt.figure(figsize=(7, 4))
plt.plot(range(1, H + 1), mlp_rmse_h,  'o-', label='MLP')
plt.plot(range(1, H + 1), lstm_rmse_h, 's-', label='LSTM')
plt.xlabel('예측 horizon (일)'); plt.ylabel('RMSE (℃)')
plt.title('Horizon별 RMSE (예측이 멀어질수록 오차 증가 경향)')
plt.legend(); plt.grid(True); plt.show()

## 8. 정리 및 확장 아이디어

이 튜토리얼에서는 다음 흐름을 익혔습니다.

1. **데이터 로드**: GitHub의 공개 CSV를 `pandas` 로 직접 읽어 `Date` 인덱스 + 수치 컬럼 형태로 다룸
2. **데이터 표현**: 원본 시계열은 `(T, D)`, 신경망 학습용으로는 `(N, L, D)` / `(N, H, D)` 형태로 변환
3. **정규화**: 학습 구간에서 통계를 계산해 누수 방지
4. **슬라이딩 윈도우**로 입/출력 페어를 구성 (multi-step, H=7)
5. **Dataset / DataLoader**로 미니배치 학습 파이프라인 구축
6. **MLP vs LSTM** 두 가지 모델을 비교
7. **예측 시각화 (t+1, 7일 궤적) + horizon별 RMSE** 정량 평가

### 확장해볼 만한 것들
- 입력 길이 `L` 이나 예측 horizon `H` 를 바꿔가며 성능 변화 관찰
- **추가 feature**: 요일(day-of-week), 월(month), 계절 등 시간 파생 변수
- **1D CNN**, **Transformer** 등 다른 아키텍처와 비교
- 다변량 데이터 (예: Jena Climate) 로 확장 — 코드 변경 거의 없음 (D 값만 달라짐)
- **Autoregressive 추론** vs **Direct multi-step** (현재 방식) 비교